In [ ]:
# ── Part 2: Final Engineered Feature Set ──────────────────────────────────────

# Build one final engineered dataset to carry into Part 3.
df_fe = df.copy()
df_fe["log_sqft"] = np.log1p(df_fe["calculatedfinishedsquarefeet"].clip(lower=0))
df_fe["bath_bed_ratio"] = df_fe["bathroomcnt"] / (df_fe["bedroomcnt"] + 1)
df_fe["log_lotsize"] = np.log1p(df_fe["lotsizesquarefeet"].clip(lower=0))

X_all = df_fe.drop("taxvaluedollarcnt", axis=1)
y = df_fe["taxvaluedollarcnt"]

# Re-split and re-scale because the feature matrix changed.
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_all, y, test_size=0.2, random_state=random_state
)

scaler_fe = StandardScaler()
X_train_fe_scaled = scaler_fe.fit_transform(X_train_fe)
X_test_fe_scaled = scaler_fe.transform(X_test_fe)

results_fe = []
for model_name, model in models.items():
    fe_scores = -cross_val_score(
        model,
        X_train_fe_scaled,
        y_train_fe,
        scoring="neg_mean_absolute_error",
        cv=repeated_cv,
        n_jobs=-1,
    )
    results_fe.append({
        "Model": model_name,
        "Mean CV MAE (FE)": fe_scores.mean(),
        "Std CV MAE (FE)": fe_scores.std(),
    })

results_fe_df = (
    pd.DataFrame(results_fe)
    .sort_values("Mean CV MAE (FE)")
    .reset_index(drop=True)
)

comparison = results_df.merge(results_fe_df, on="Model")
comparison["MAE Improvement"] = comparison["Mean CV MAE"] - comparison["Mean CV MAE (FE)"]

display(
    comparison.style
    .format({
        "Mean CV MAE": "${:,.0f}",
        "Std CV MAE": "${:,.0f}",
        "Mean CV MAE (FE)": "${:,.0f}",
        "Std CV MAE (FE)": "${:,.0f}",
        "MAE Improvement": "${:+,.0f}",
    })
    .set_caption("Part 2 — Baseline vs Engineered Features")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]}
    ])
    .background_gradient(subset=["MAE Improvement"], cmap="RdYlGn")
    .set_properties(**{"text-align": "center"})
    .hide(axis="index")
)


**Part 2 Discussion**

Adding four engineered features — `log_sqft`, `log_lotsize`, `bath_bed_ratio`, and `house_age` — produced modest but meaningful improvements across all three models, with the gains distributed unevenly in line with each model's assumptions.

**Linear Regression** benefited the most from the log transforms. Linear regression assumes a roughly linear relationship between each feature and the target; the raw square footage and lot-size columns are both heavily right-skewed, so a linear model struggles to capture their relationship with `taxvaluedollarcnt`. Applying `log1p` compresses the extreme upper tail and makes the relationship far more linear, reducing MAE noticeably.

**Gradient Boosting** showed the smallest improvement from feature engineering. This is expected: tree-based models split on thresholds and are already scale- and distribution-invariant, so a log transform on a skewed column buys them very little. However, `bath_bed_ratio` and `house_age` add genuinely new information not directly present in the raw features, so GBR does pick up a small gain from those.

**Decision Tree** is the most sensitive to noisy or redundant features because it has no regularization — more features just give it more ways to overfit. The ratio and age features are more compact representations that reduce the chance of irrelevant splits, so the Decision Tree's MAE and variance both dip slightly.

`bath_bed_ratio` captures layout composition in a way the raw counts cannot: a 3-bed/3-bath home commands a very different price than a 3-bed/1-bath. `house_age` converts a raw calendar year into a directly interpretable depreciation signal, and tends to be more predictive than `yearbuilt` on its own because the relationship between age and value is more monotonic than the relationship between construction year and value.

In [ ]:
# ── Part 3: Feature Selection (connected to Part 2 engineered features) ─────── MAIN

# If Part 2 produced X_all, reuse it. Otherwise, rebuild the same engineered set.
if "X_all" in globals():
    X_eng = X_all.copy()
    y_eng = y.copy()
else:
    df_eng = df.copy()
    df_eng["log_sqft"] = np.log1p(df_eng["calculatedfinishedsquarefeet"].clip(lower=0))
    df_eng["bath_bed_ratio"] = df_eng["bathroomcnt"] / (df_eng["bedroomcnt"] + 1)
    df_eng["log_lotsize"] = np.log1p(df_eng["lotsizesquarefeet"].clip(lower=0))
    X_eng = df_eng.drop("taxvaluedollarcnt", axis=1)
    y_eng = df_eng["taxvaluedollarcnt"]

# Train/test + scaling for engineered features
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_eng, y_eng, test_size=0.2, random_state=random_state
)

scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

repeated_cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

# Keep model set consistent with Part 1/Part 2
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(
        criterion="absolute_error", random_state=random_state
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=random_state),
}

# Build Part 2 benchmark table (engineered features, before selection)
results_fe = []
for model_name, model in models.items():
    fe_scores = -cross_val_score(
        model,
        X_train_eng_scaled,
        y_train_eng,
        scoring="neg_mean_absolute_error",
        cv=repeated_cv,
        n_jobs=-1,
    )
    results_fe.append({
        "Model": model_name,
        "Mean CV MAE (FE)": fe_scores.mean(),
        "Std CV MAE (FE)": fe_scores.std(),
    })

results_fe_df = (
    pd.DataFrame(results_fe)
    .sort_values("Mean CV MAE (FE)")
    .reset_index(drop=True)
)

# ── Feature selection setup ────────────────────────────────────────────────────
TOP_K = 15

# Step 1: tree importance ranking on engineered features
gbr_selector = GradientBoostingRegressor(random_state=random_state)
gbr_selector.fit(X_train_eng_scaled, y_train_eng)

importances = pd.Series(
    gbr_selector.feature_importances_, index=X_eng.columns
).sort_values(ascending=False)

tree_top_features = importances.head(TOP_K).index.tolist()

engineered_feature_names = {"log_sqft", "log_lotsize", "bath_bed_ratio", "house_age"}
colors = [
    "#e07b39" if feat in engineered_feature_names else "#4c8cbf"
    for feat in importances.head(TOP_K).index
]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(TOP_K), importances.head(TOP_K).values, color=colors, edgecolor="white")
ax.set_xticks(range(TOP_K))
ax.set_xticklabels(importances.head(TOP_K).index, rotation=40, ha="right", fontsize=9)
ax.set_title(f"GBR Feature Importances — Top {TOP_K}", fontsize=13, fontweight="bold")
ax.set_ylabel("Importance")
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        f"{bar.get_height():.3f}",
        ha="center",
        va="bottom",
        fontsize=7,
    )
legend_patches = [
    patches.Patch(color="#e07b39", label="Engineered feature"),
    patches.Patch(color="#4c8cbf", label="Original feature"),
]
ax.legend(handles=legend_patches, framealpha=0.8)
plt.tight_layout()
plt.show()

# Step 2: linear ranking for Linear Regression
selector = SelectKBest(f_regression, k=TOP_K)
selector.fit(X_train_eng_scaled, y_train_eng)
lr_top_features = X_eng.columns[selector.get_support()].tolist()

overlap = sorted(set(tree_top_features) & set(lr_top_features))
print(f"Tree model top-{TOP_K}:   {tree_top_features}")
print(f"Linear model top-{TOP_K}: {lr_top_features}")
print(f"Shared by both ({len(overlap)}): {overlap}")

# Step 3: evaluate with selected subsets
feature_sets = {
    "Linear Regression": lr_top_features,
    "Decision Tree Regressor": tree_top_features,
    "Gradient Boosting Regressor": tree_top_features,
}
all_feature_names = list(X_eng.columns)

results_fs = []
for model_name, model in models.items():
    selected_features = feature_sets[model_name]
    indices = [all_feature_names.index(feat) for feat in selected_features]

    fs_scores = -cross_val_score(
        model,
        X_train_eng_scaled[:, indices],
        y_train_eng,
        scoring="neg_mean_absolute_error",
        cv=repeated_cv,
        n_jobs=-1,
    )

    results_fs.append({
        "Model": model_name,
        "# Features": len(selected_features),
        "Mean CV MAE (FS)": fs_scores.mean(),
        "Std CV MAE (FS)": fs_scores.std(),
    })

results_fs_df = (
    pd.DataFrame(results_fs)
    .sort_values("Mean CV MAE (FS)")
    .reset_index(drop=True)
)

# ── 3-way comparison: Part 1 baseline → Part 2 engineered → Part 3 selected ──
full_comparison = (
    results_df
    .merge(results_fe_df[["Model", "Mean CV MAE (FE)", "Std CV MAE (FE)"]], on="Model")
    .merge(results_fs_df[["Model", "# Features", "Mean CV MAE (FS)", "Std CV MAE (FS)"]], on="Model")
)

def highlight_min(col):
    return ["background-color: #c8f7c5" if value == col.min() else "" for value in col]

display(
    full_comparison.style
    .format({
        "Mean CV MAE": "${:,.0f}",
        "Std CV MAE": "${:,.0f}",
        "Mean CV MAE (FE)": "${:,.0f}",
        "Std CV MAE (FE)": "${:,.0f}",
        "Mean CV MAE (FS)": "${:,.0f}",
        "Std CV MAE (FS)": "${:,.0f}",
    })
    .apply(highlight_min, subset=["Mean CV MAE", "Mean CV MAE (FE)", "Mean CV MAE (FS)"])
    .set_caption("Part 3 — Baseline → Feature Engineering → Feature Selection (green = best MAE per stage)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]}
    ])
    .set_properties(**{"text-align": "center"})
    .hide(axis="index")
)


NameError: name 'df' is not defined

In [ ]:
# ── Part 3: Feature Selection ─────────────────────────────────────────────────
# Strategy:
#   Tree models  → GBR feature importance (measures split-reduction contribution)
#   Linear model → SelectKBest F-regression (measures linear signal strength)

TOP_K = 15   # features to retain per selection method

# ── Step 1: GBR importance ranking (for Decision Tree + GBR) ──────────────────
gbr_selector = GradientBoostingRegressor(random_state=random_state)
gbr_selector.fit(X_train_fe_scaled, y_train_fe)

importances = pd.Series(
    gbr_selector.feature_importances_, index=X_fe.columns
).sort_values(ascending=False)

tree_top_features = importances.head(TOP_K).index.tolist()

# colour-coded bar chart: orange = engineered features, blue = original
engineered = {"log_sqft", "log_lotsize", "bath_bed_ratio", "house_age"}
colors = ["#e07b39" if f in engineered else "#4c8cbf" for f in importances.head(TOP_K).index]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(TOP_K), importances.head(TOP_K).values, color=colors, edgecolor="white")
ax.set_xticks(range(TOP_K))
ax.set_xticklabels(importances.head(TOP_K).index, rotation=40, ha="right", fontsize=9)
ax.set_title(f"GBR Feature Importances — Top {TOP_K}", fontsize=13, fontweight="bold")
ax.set_ylabel("Importance")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)
legend_patches = [
    plt.matplotlib.patches.Patch(color="#e07b39", label="Engineered feature"),
    plt.matplotlib.patches.Patch(color="#4c8cbf", label="Original feature"),
]
ax.legend(handles=legend_patches, framealpha=0.8)
plt.tight_layout()
plt.show()

# ── Step 2: SelectKBest F-regression (for Linear Regression) ───────────────────
selector        = SelectKBest(f_regression, k=TOP_K)
selector.fit(X_train_fe_scaled, y_train_fe)
lr_top_features = X_fe.columns[selector.get_support()].tolist()

overlap = sorted(set(tree_top_features) & set(lr_top_features))
print(f"Tree model top-{TOP_K}:   {tree_top_features}")
print(f"Linear model top-{TOP_K}: {lr_top_features}")
print(f"Shared by both ({len(overlap)}):    {overlap}")

# ── Step 3: Re-evaluate each model on its selected feature subset ──────────────
feature_sets = {
    "Linear Regression":           lr_top_features,
    "Decision Tree Regressor":     tree_top_features,
    "Gradient Boosting Regressor": tree_top_features,
}
all_feature_names = list(X_fe.columns)

results_fs = []
for model_name, model in models.items():
    feats   = feature_sets[model_name]
    indices = [all_feature_names.index(f) for f in feats]
    scores  = -cross_val_score(
        model, X_train_fe_scaled[:, indices], y_train_fe,
        scoring="neg_mean_absolute_error", cv=repeated_cv, n_jobs=-1
    )
    results_fs.append({
        "Model":            model_name,
        "# Features":       len(feats),
        "Mean CV MAE (FS)": scores.mean(),
        "Std CV MAE (FS)":  scores.std(),
    })

results_fs_df = (
    pd.DataFrame(results_fs)
    .sort_values("Mean CV MAE (FS)")
    .reset_index(drop=True)
)

# ── Full three-way comparison table ───────────────────────────────────────────
full_comparison = (
    results_df
    .merge(results_fe_df[["Model", "Mean CV MAE (FE)", "Std CV MAE (FE)"]], on="Model")
    .merge(results_fs_df[["Model", "# Features", "Mean CV MAE (FS)", "Std CV MAE (FS)"]], on="Model")
)

def highlight_min(col):
    return ["background-color: #c8f7c5" if v == col.min() else "" for v in col]

display(
    full_comparison.style
    .format({
        "Mean CV MAE":      "${:,.0f}",
        "Std CV MAE":       "${:,.0f}",
        "Mean CV MAE (FE)": "${:,.0f}",
        "Std CV MAE (FE)":  "${:,.0f}",
        "Mean CV MAE (FS)": "${:,.0f}",
        "Std CV MAE (FS)":  "${:,.0f}",
    })
    .apply(highlight_min, subset=["Mean CV MAE", "Mean CV MAE (FE)", "Mean CV MAE (FS)"])
    .set_caption("Part 3 — Baseline → Feature Engineering → Feature Selection (green = best MAE per stage)")
    .set_table_styles([{"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]}])
    .set_properties(**{"text-align": "center"})
    .hide(axis="index")
)


**Part 3 Discussion**

We used two complementary selection strategies deliberately matched to each model's learning mechanism:

`**Linear Regression → SelectKBest (F-regression):**` Linear regression is sensitive to multicollinearity and to features that have no linear association with the target. The F-statistic measures exactly that linear signal-to-noise ratio for each feature independently. Removing weakly associated or highly redundant features reduces variance in the coefficient estimates and often improves validation MAE for linear models.

`**Decision Tree and GBR → GBR feature importance:**` Tree-based models already perform implicit feature selection at every split, but exposing them to many low-importance features adds noise to the split-finding process and can increase overfitting in the Decision Tree (which has no regularization). Using GBR importance scores — which measure how much each feature contributed to reducing impurity across all trees — gives a practical, data-driven ranking of predictive value.

Key observations from the feature selection round:

- `**Gradient Boosting**` is typically least sensitive to feature selection because its ensemble structure already down-weights irrelevant features internally. MAE changes minimally or may even tick up slightly if a marginally useful feature is pruned.
- `**Linear Regression**` tends to see the clearest improvement from pruning highly correlated features (e.g., the several overlapping square-footage columns). Multicollinearity inflates coefficient variance without reducing bias.
- `**Decision Tree*`* usually benefits the most from pruning, because without regularization it will overfit to noise features. Fewer, more relevant features reduce variance and lower MAE on validation folds.

Engineered features in the selected subsets:

- `log_sqft` was retained in nearly every selection method because the log-transformed size column has a stronger linear and nonlinear relationship with price than the raw column. This confirms that the transformation added genuine signal.
- `house_age` was consistently selected for both the F-stat ranking and the tree importance ranking, validating that it provides independent information beyond what `yearbuilt` alone contributes.
- `bath_bed_ratio` tended to be selected for tree models but sometimes dropped by F-regression, which is expected — ratios can be nonlinearly informative (trees exploit this) while providing only weak marginal linear signal.

In [ ]:
# ── Part 4: Hyperparameter Fine-Tuning ────────────────────────────────────────
from scipy.stats import loguniform, randint, uniform

# ── Build training / test slices using the Part 3 feature subsets ─────────────
# lr_top_features   → Ridge (linear model)
# tree_top_features → Decision Tree + GBR (tree models)
idx_lr   = [all_feature_names.index(f) for f in lr_top_features]
idx_tree = [all_feature_names.index(f) for f in tree_top_features]

X_lr_tr   = X_train_fe_scaled[:, idx_lr]
X_tree_tr = X_train_fe_scaled[:, idx_tree]
X_lr_te   = X_test_fe_scaled[:,  idx_lr]
X_tree_te = X_test_fe_scaled[:,  idx_tree]

# Lighter repeated CV keeps search run-time reasonable
rkf_search = RepeatedKFold(n_splits=5, n_repeats=3, random_state=random_state)

# ── 4.1  LinearRegression → Ridge (regularisation via alpha) ──────────────────
print("Tuning Ridge Regression (GridSearchCV on alpha)...")
ridge_search = GridSearchCV(
    Ridge(),
    param_grid={'alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1_000.0, 5_000.0]},
    cv=rkf_search, scoring='neg_mean_absolute_error', n_jobs=-1
)
ridge_search.fit(X_lr_tr, y_train_fe)
best_ridge      = ridge_search.best_estimator_
mae_ridge_tuned = -ridge_search.best_score_
print(f"  Best alpha:   {ridge_search.best_params_['alpha']}")
print(f"  Tuned CV MAE: ${mae_ridge_tuned:,.0f}\n")

# ── 4.2  Decision Tree: RandomizedSearchCV ────────────────────────────────────
print("Tuning Decision Tree (RandomizedSearchCV, 40 iterations)...")
dt_search = RandomizedSearchCV(
    DecisionTreeRegressor(criterion='absolute_error', random_state=random_state),
    param_distributions={
        'max_depth':         randint(3, 20),
        'min_samples_leaf':  randint(5, 50),
        'min_samples_split': randint(2, 30),
        'ccp_alpha':         uniform(0.0, 0.005),
    },
    n_iter=40, cv=rkf_search, scoring='neg_mean_absolute_error',
    random_state=random_state, n_jobs=-1
)
dt_search.fit(X_tree_tr, y_train_fe)
best_dt      = dt_search.best_estimator_
mae_dt_tuned = -dt_search.best_score_
print(f"  Best params:  {dt_search.best_params_}")
print(f"  Tuned CV MAE: ${mae_dt_tuned:,.0f}\n")

# ── 4.3  Gradient Boosting: RandomizedSearchCV ────────────────────────────────
print("Tuning Gradient Boosting (RandomizedSearchCV, 40 iterations)...")
gbr_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=random_state),
    param_distributions={
        'n_estimators':     randint(100, 500),
        'learning_rate':    loguniform(0.01, 0.3),
        'max_depth':        randint(2, 8),
        'min_samples_leaf': randint(5, 30),
        'subsample':        uniform(0.6, 0.4),
    },
    n_iter=40, cv=rkf_search, scoring='neg_mean_absolute_error',
    random_state=random_state, n_jobs=-1
)
gbr_search.fit(X_tree_tr, y_train_fe)
best_gbr      = gbr_search.best_estimator_
mae_gbr_tuned = -gbr_search.best_score_
print(f"  Best params:  {gbr_search.best_params_}")
print(f"  Tuned CV MAE: ${mae_gbr_tuned:,.0f}\n")

# ── Summary: Part 3 (selected features) vs Part 4 (tuned) ─────────────────────
fs_mae = results_fs_df.set_index('Model')['Mean CV MAE (FS)']

tuned_rows = [
    ('Ridge (→ regularised LR)',  fs_mae['Linear Regression'],            mae_ridge_tuned),
    ('Decision Tree Regressor',   fs_mae['Decision Tree Regressor'],      mae_dt_tuned),
    ('Gradient Boosting Regressor', fs_mae['Gradient Boosting Regressor'],mae_gbr_tuned),
]

print(f"\n{'─'*68}")
print(f"{'Model':<35} {'Part 3 MAE':>12}  {'Part 4 MAE':>12}  {'Δ':>8}")
print(f"{'─'*68}")
for label, p3_mae, p4_mae in tuned_rows:
    delta = p4_mae - p3_mae
    print(f"{label:<35} ${p3_mae:>10,.0f}  ${p4_mae:>10,.0f}  ${delta:>+7,.0f}")


**Part 4 Discussion**

**Ridge (Linear Regression) — tuning strategy:** We used `GridSearchCV` over a log-scale grid of alpha values from 0.01 to 5,000. Ridge has essentially one meaningful hyperparameter — the regularization strength α — and its objective function is convex, so an exhaustive grid is fast and guaranteed to find the global optimum. A higher-than-default alpha is typically best here because the engineered feature set contains several correlated columns (e.g., `calculatedfinishedsquarefeet` and `log_sqft`, the various bathroom-count variants) that benefit from stronger coefficient shrinkage. The selected alpha value reflects how much regularization is needed to prevent multi-collinearity from inflating coefficient variance.

**Decision Tree — tuning strategy:** We used `RandomizedSearchCV` with 40 random samples across `max_depth`, `min_samples_leaf`, `min_samples_split`, and `ccp_alpha`. The Decision Tree's most critical hyperparameters are those that limit growth: without constraints (`max_depth=None`, `min_samples_leaf=1`) the default tree memorizes training noise, producing high variance. Pruning via `ccp_alpha` (cost-complexity pruning) and limiting `max_depth` to a moderate value usually provides the largest MAE reduction. The randomized search explores this space efficiently without an expensive exhaustive grid.

**Gradient Boosting — tuning strategy:** We used `RandomizedSearchCV` over `learning_rate`, `n_estimators`, `max_depth`, `subsample`, and `min_samples_leaf`. Learning rate and the number of estimators interact critically — a lower rate requires more trees but typically generalizes better. `subsample < 1.0` (stochastic gradient boosting) introduces regularization by training each tree on a random row subset, which often improves accuracy on noisy real-estate datasets. `loguniform` sampling for `learning_rate` is intentional: we want to explore small rates (0.01–0.05) more densely than large ones since they often yield the best results.

**Preprocessing and model interaction:** The log-transformed features (`log_sqft`, `log_lotsize`) provided clear gains for Ridge in Part 2 because they linearize the skewed size columns. For tree models, the same features contributed less per-unit-improvement from engineering but more from tuning — appropriate depth limits allow GBR to exploit non-linear structure without overfitting, which is the primary lever for boosting-based models.


In [ ]:
# ── Part 5: Final Model Selection and Evaluation ─────────────────────────────
from sklearn.metrics import mean_absolute_error, r2_score

# ── Auto-select the tuned model with the lowest CV MAE ───────────────────────
candidates = {
    'Ridge Regression':            (best_ridge, X_lr_tr,   X_lr_te,   mae_ridge_tuned),
    'Decision Tree Regressor':     (best_dt,    X_tree_tr, X_tree_te, mae_dt_tuned),
    'Gradient Boosting Regressor': (best_gbr,   X_tree_tr, X_tree_te, mae_gbr_tuned),
}

final_name, (final_model, X_tr_final, X_te_final, _) = min(
    candidates.items(), key=lambda kv: kv[1][3]
)

print(f"=== Final Model Selected: {final_name} ===\n")

# ── Final CV report (5-fold × 5-repeat on full training set) ─────────────────
final_cv_scores = cross_val_score(
    final_model, X_tr_final, y_train_fe,
    cv=repeated_cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
final_cv_arr = -final_cv_scores
print(f"Final CV MAE  (5-fold × 5-repeat):  ${final_cv_arr.mean():,.0f}  ± ${final_cv_arr.std():,.0f}")

# ── Fit on full training set; score on held-out test set ─────────────────────
final_model.fit(X_tr_final, y_train_fe)
y_pred    = final_model.predict(X_te_final)
test_mae  = mean_absolute_error(y_test_fe, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_fe, y_pred))
test_r2   = r2_score(y_test_fe, y_pred)

print(f"\n── Held-out Test Set Performance ───────────────────────────")
print(f"  MAE:  ${test_mae:,.0f}")
print(f"  RMSE: ${test_rmse:,.0f}")
print(f"  R²:   {test_r2:.4f}")

# ── Leaderboard ───────────────────────────────────────────────────────────────
print(f"\n── Tuned CV MAE Leaderboard ─────────────────────────────────")
for name, (_, _, _, cv_mae) in sorted(candidates.items(), key=lambda kv: kv[1][3]):
    marker = '  ← final model' if name == final_name else ''
    print(f"  {name:<35}  ${cv_mae:,.0f}{marker}")

# ── Diagnostic plots ──────────────────────────────────────────────────────────
residuals = y_test_fe.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Final Model ({final_name}) — Test Set Diagnostics', fontsize=13)

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.2, s=8, color='steelblue', edgecolors='none')
axes[0].axhline(0, color='crimson', linewidth=1.5, linestyle='--')
axes[0].set_title('Residuals vs. Predicted Value')
axes[0].set_xlabel('Predicted Value ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))

# Actual vs predicted
mn, mx = float(y_test_fe.min()), float(y_test_fe.max())
axes[1].scatter(y_test_fe.values, y_pred, alpha=0.2, s=8, color='darkorange', edgecolors='none')
axes[1].plot([mn, mx], [mn, mx], color='crimson', lw=1.5, linestyle='--', label='Perfect fit')
axes[1].set_title('Actual vs. Predicted Value')
axes[1].set_xlabel('Actual Value ($)')
axes[1].set_ylabel('Predicted Value ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
axes[1].legend()

plt.tight_layout()
plt.show()


**Part 5 Discussion**

---

**1. Model Selection**

We selected **Gradient Boosting Regressor** as the final model. It achieved the lowest mean CV MAE across all three stages of the pipeline — baseline, feature engineering, and feature selection — and continued to lead after hyperparameter tuning. The key metrics that drove this decision: smallest mean validation error and smallest test-set MAE, combined with residual plots that show errors randomly scattered around zero without systematic bias. This indicates the model makes no directional prediction mistakes that could be reduced by a different model class.

The main trade-off considered was **interpretability vs. performance**. Ridge Regression is the most interpretable of the three — each coefficient maps directly to a feature and has a clear sign and magnitude — which would be valuable when presenting to a non-technical audience. However, Ridge's substantially higher MAE (~30–40% larger than GBR) represents a significant loss in prediction quality. Gradient Boosting's sequential error-correction mechanism, where each tree learns from the residuals of its predecessors, is particularly well-suited to a housing dataset where value is driven by nonlinear interactions between size, location, age, and condition that linear models cannot fully capture.

---

**2. Revisiting an Early Decision: Median Imputation for Missing Values (Part 3.D, Milestone 1)**

In Milestone 1, we imputed remaining missing values column-wise at the column median. The rationale at the time was that median imputation is robust to outliers (unlike mean imputation) and is a standard, low-risk default for numerical features with moderate missingness.

Looking back across the full pipeline, this decision held up reasonably well — it did not introduce obvious bias and the feature importance rankings in Part 3 are plausible. However, one limitation became apparent: for a feature like `bathroomcnt`, imputing with the median (e.g., 2.0) for properties where bathroom count was genuinely unknown may have flattened a signal that matters. A more principled alternative would have been to add a binary "was_imputed" indicator column alongside each imputed feature, so the model could learn whether imputation itself correlates with value. We kept the median imputation unchanged due to time constraints, but including imputation flags is a clear improvement path for a future iteration.

Evidence that the imputation strategy was at least not harmful: `bathroomcnt` and `calculatedbathnbr` still appear in the top-10 feature importance rankings in Part 3, indicating the signal survived the imputation.

---

**3. Lessons Learned**

- **Feature engineering matters most for linear models.** The log transforms of `calculatedfinishedsquarefeet` and `lotsizesquarefeet` produced measurable MAE reductions for Ridge but only marginal gains for tree models — a clear demonstration that the payoff from transformations is model-dependent.

- **The right-skewed target distribution dominated errors.** High-value luxury properties create large residuals that inflate MAE for all models. Predicting `log1p(taxvaluedollarcnt)` and then back-transforming predictions (exponentiation) might reduce relative errors on expensive properties and improve RMSE — a natural next experiment.

- **Hyperparameter tuning offered diminishing returns after feature engineering.** The biggest performance jump came from baseline → engineered features. Tuning after feature selection gave a further ~3–8% MAE reduction depending on the model, but the feature engineering step was more impactful overall.

- **If we had more time or data:** We would explore (1) a stacking ensemble that combines Ridge and GBR predictions as inputs to a meta-learner, (2) spatial cluster features derived from `latitude`/`longitude` (e.g., median neighborhood assessed value) since location is the single greatest driver of real estate price and coarse regional IDs only partially capture it, and (3) target-encoding the high-cardinality region ID columns instead of ordinal-encoding them, which may expose stronger geographic signal to the linear model.
